In [1]:

import os
import glob
import json
import csv
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.utils import load_img, img_to_array

# -----------------------------------------""
# 1. PATHS
# -----------------------------------------

PROJECT_PATH = r"C:\Users\DELL\OneDrive\Downloads\Vision_Based_Sorting_Project"

MODEL_PATH = os.path.join(
    PROJECT_PATH, "trained_waste_model", "waste_classifier.keras"
)

CLASS_PATH = os.path.join(
    PROJECT_PATH, "trained_waste_model", "class_names.json"
)

TEST_FOLDER = os.path.join(PROJECT_PATH, "test_images")

# -----------------------------------------
# 2. LOAD MODEL AND CLASS NAMES
# -----------------------------------------

model = tf.keras.models.load_model(MODEL_PATH)

with open(CLASS_PATH, "r") as f:
    class_names = json.load(f)

print("Model loaded successfully!")
print("Classes:", class_names)

# -----------------------------------------
# 3. RECYCLABILITY MAPPING
# -----------------------------------------

# Practical example: adjust to your local recycling rules.
recyclability = {
    "battery": "Recyclable (special collection)",
    "biological": "Compostable / Organic",
    "brown-glass": "Recyclable",
    "cardboard": "Recyclable",
    "clothes": "Reusable / Textile recycling",
    "green-glass": "Recyclable",
    "metal": "Recyclable",
    "paper": "Recyclable",
    "plastic": "Recyclable (depends on type)",
    "shoes": "Reusable / Textile recycling",
    "trash": "Usually non-recyclable"
}

# -----------------------------------------
# 4. FIND IMAGES
# -----------------------------------------

extensions = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]
image_files = []

for ext in extensions:
    image_files.extend(
        glob.glob(os.path.join(TEST_FOLDER, "**", ext),
                  recursive=True)
    )

image_files = sorted(set(image_files))

if not image_files:
    print("No images found. Check your test_images path.")

else:
    print("\nTotal images found:", len(image_files))
    results = []

    # -------------------------------------
    # 5. PREDICT EACH IMAGE
    # -------------------------------------

    for image_path in image_files:
        try:
            img = load_img(
                image_path,
                target_size=(224, 224),
                color_mode="rgb"
            )

            img_array = img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0)

            predictions = model.predict(img_array, verbose=0)[0]

            index = int(np.argmax(predictions))
            predicted_class = class_names[index]
            confidence = float(predictions[index]) * 100

            waste_status = recyclability.get(
                predicted_class,
                "Needs manual checking"
            )

            print("\nImage:", os.path.basename(image_path))
            print("Waste category:", predicted_class)
            print("Recycling status:", waste_status)
            print("Confidence:", round(confidence, 2), "%")

            results.append({
                "Image": os.path.basename(image_path),
                "Waste Category": predicted_class,
                "Recycling Status": waste_status,
                "Confidence (%)": round(confidence, 2)
            })

        except Exception as e:
            print("Error processing", image_path, ":", e)

    # -------------------------------------
    # 6. SAVE RESULTS
    # -------------------------------------

    if results:
        csv_path = os.path.join(
            PROJECT_PATH, "test_results.csv"
        )

        with open(
            csv_path, "w", newline="", encoding="utf-8"
        ) as f:
            writer = csv.DictWriter(
                f, fieldnames=results[0].keys()
            )
            writer.writeheader()
            writer.writerows(results)

        print("\nTesting completed!")
        print("Images tested:", len(results))
        print("Results saved to:", csv_path)


Model loaded successfully!
Classes: ['battery', 'biological', 'brown-glass', 'clothes', 'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass']

Total images found: 4

Image: Georgian Glass - Antique Drinking Glasses -18th century english plain stem glasses.jpg
Waste category: white-glass
Recycling status: Needs manual checking
Confidence: 100.0 %

Image: metal95.jpg
Waste category: metal
Recycling status: Recyclable
Confidence: 69.09 %

Image: paper971.jpg
Waste category: paper
Recycling status: Recyclable
Confidence: 99.95 %

Image: plastic-bottles-photo.jpg
Waste category: plastic
Recycling status: Recyclable (depends on type)
Confidence: 99.95 %

Testing completed!
Images tested: 4
Results saved to: C:\Users\DELL\OneDrive\Downloads\Vision_Based_Sorting_Project\test_results.csv
